In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [2]:
from src.data.filter import load_data, filter_with_nation_winners
from src.data.preprocess import process_first_innings, process_second_innings, get_label

data = load_data()
data = filter_with_nation_winners(data)

outcomes = [get_label(d) for d in data]
first_innings = [process_first_innings(d) for d in data]
second_innings = [process_second_innings(d) for d in data]

len(outcomes), len(first_innings), len(second_innings)

(2040, 2040, 2040)

In [3]:
def get_sequential_data(innings, outcomes):
    sequential_data = []
    for inning, outcome in zip(innings, outcomes):
        sequential_data.append({
            'innings': [list(state.values()) for state in inning],
            'labels': outcome
        })
    return sequential_data
    

In [4]:
first_inning_seq = get_sequential_data(first_innings, outcomes)
second_inning_seq = get_sequential_data(second_innings, outcomes)

In [5]:
import torch
import torch.nn as nn

def collator(batch):
    innings = [torch.tensor(item['innings'], dtype=torch.float32) for item in batch]
    loss_mask = [torch.ones(len(item['innings']), dtype=torch.float32) for item in batch]
    labels = [torch.tensor(item['labels'], dtype=torch.float32) for item in batch]

    innings = nn.utils.rnn.pad_sequence(innings, batch_first=True, padding_value=-1.0)
    loss_mask = nn.utils.rnn.pad_sequence(loss_mask, batch_first=True, padding_value=0.0)
    labels = torch.stack(labels).reshape(-1, 1).repeat(1, innings.shape[1])

    return {'innings': innings, 'labels': labels, 'loss_mask': loss_mask}

In [6]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(LSTMModel, self).__init__()
        self.fc_input = nn.Linear(3, input_size)  # Adjust input size to match the LSTM input
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, num_layers=num_layers)
        self.fc = nn.Linear(hidden_size, 1)
        self.loss_function = nn.BCEWithLogitsLoss(reduction='none')  

    def compute_loss(self, output, labels, loss_mask):
        loss = self.loss_function(output, labels)
        masked_loss = loss * loss_mask
        return masked_loss.sum() / loss_mask.sum()

    def forward(self, innings, loss_mask, labels):
        innings = self.fc_input(innings)
        lstm_out, _ = self.lstm(innings)
        output = self.fc(lstm_out)
        loss = self.compute_loss(output.squeeze(-1), labels, loss_mask)
        return {'loss': loss, 'logits': output.squeeze(-1)}


In [7]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(second_inning_seq, test_size=0.2, random_state=42)

In [11]:
from transformers import Trainer, TrainingArguments

input_size = 16
hidden_size = 32
num_layers = 4
batch_size = 32

train_args = TrainingArguments(
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=512,
    num_train_epochs=40,
    learning_rate=1e-3,
    output_dir=str(project_root / "models" / "simple_model"),
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    remove_unused_columns=False,
 )

model = LSTMModel(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=collator,
)

trainer.train()


Step,Training Loss,Validation Loss
100,0.560327,0.528771
200,0.459790,0.512114
300,0.456206,0.475469
400,0.435901,0.480049
500,0.428844,0.483573
600,0.427800,0.449433
700,0.419450,0.457401
800,0.421115,0.446972
900,0.413486,0.443457
1000,0.413966,0.439804


TrainOutput(global_step=2040, training_loss=0.4172934144150977, metrics={'train_runtime': 40.9306, 'train_samples_per_second': 1594.894, 'train_steps_per_second': 49.84, 'total_flos': 0.0, 'train_loss': 0.4172934144150977, 'epoch': 40.0})